# Vector Calculus

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 02.08 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/09_vector_calculus.ipynb)

---

## 🎯 Learning Objective

Learn divergence, curl, and the Laplacian as tools for describing vector fields, and preview Green's and Stokes' theorems.

---

## 📐 Theory

A **vector field** $\mathbf{F}(x,y) = (P(x,y), Q(x,y))$ assigns a vector to every point in
space — think of it as describing a flow, like wind velocity at every point on a map. Vector
calculus gives you scalar and vector tools to describe how such a flow behaves locally.

### Divergence: is the flow expanding or contracting?

The **divergence** of $\mathbf{F}$ is a scalar field:

$$\text{div}\,\mathbf{F} = \nabla\cdot\mathbf{F} = \frac{\partial P}{\partial x} + \frac{\partial Q}{\partial y}$$

At a point, $\text{div}\,\mathbf{F} > 0$ means the flow is a net "source" there (more flow
leaves a tiny neighborhood than enters — like a point emitting fluid); $\text{div}\,\mathbf{F}<0$
means a net "sink." The notation $\nabla\cdot\mathbf{F}$ treats $\nabla=(\partial/\partial x,
\partial/\partial y)$ as a vector of operators and takes its formal dot product with
$\mathbf{F}$ — a mnemonic that also explains the gradient's notation from `02.03`
($\nabla f$ is this same "vector of operators" applied to a scalar).

### Curl: is the flow rotating?

In 2D, the (scalar) curl measures local rotation:

$$\text{curl}\,\mathbf{F} = \frac{\partial Q}{\partial x} - \frac{\partial P}{\partial y}$$

A positive curl means the flow locally rotates counterclockwise around that point (imagine
dropping a tiny paddle wheel into the flow and watching which way it spins); zero curl at a
point means the field is **irrotational** there — informally, no local "swirl."

Every **conservative** field ($\mathbf{F} = \nabla f$ for some scalar $f$) is irrotational
everywhere it's defined: mixed partials commute (Clairaut, `02.03`), so
$\partial_x(\partial f/\partial y) = \partial_y(\partial f/\partial x)$ forces curl $=0$
automatically. The **converse** — curl-free implies conservative — is the direction that needs
an extra hypothesis: it holds on any **simply connected** domain (no holes for a loop to catch
on, e.g. all of $\mathbb{R}^2$ or an open disk), but is false in general. The standard
counterexample is the vortex field $\mathbf{F}=(-y,x)/(x^2+y^2)$ on the punctured plane
$\mathbb{R}^2\setminus\{0\}$: it's curl-free everywhere it's defined, yet its circulation
around any loop enclosing the origin is $2\pi \ne 0$ (verified numerically below) — impossible
for a conservative field, whose circulation around *every* closed loop must be exactly $0$ (the
fundamental theorem for line integrals: $\oint_C \nabla f\cdot d\mathbf{r} = f(\text{end})-f(\text{start})=0$
on a closed loop where $f$ is single-valued). Every field used elsewhere in this notebook is
defined on all of $\mathbb{R}^2$ (simply connected), so curl-free and conservative *do* coincide
for our purposes — just not as a blanket fact for every vector field.

### The Laplacian

Applying divergence to a gradient gives the **Laplacian**, a single scalar operator combining
all pure second derivatives:

$$\nabla^2 f = \nabla\cdot(\nabla f) = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$$

This is the trace of the Hessian from `02.06` — summing curvature along each axis rather than
reporting the full matrix. $\nabla^2 f = 0$ (Laplace's equation) characterizes functions with
no local "bumps or dips" relative to their neighborhood average — a smoothness condition that
appears throughout physics and, as we'll see below, in the mathematics of diffusion.

### Green's and Stokes' theorems (previewed)

**Green's theorem** relates a line integral around a closed curve $C$ to a double integral of
curl over the region $R$ it encloses: $\oint_C \mathbf{F}\cdot d\mathbf{r} = \iint_R
\text{curl}\,\mathbf{F}\,dA$. **Stokes' theorem** is the 3D generalization, relating a surface's
boundary circulation to the curl integrated over the surface. Both say the same conceptual
thing: total rotation around a boundary equals accumulated local rotation throughout the
interior — a "local information sums to global behavior" idea structurally similar to the
Fundamental Theorem of Calculus from `02.07`. We won't need the full theorems computationally
in this notebook, but the intuition — boundary behavior reflects an integral of interior
behavior — reappears in the score-based generative models below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting three contrasting vector fields side by side — pure outward flow (positive
divergence, zero curl), pure rotation (zero divergence, nonzero curl), and a gradient field
(irrotational by construction) — makes divergence and curl visually distinguishable at a glance.

In [ ]:
# Three canonical vector fields: pure source (div>0, curl=0), pure rotation (div=0, curl>0),
# and a gradient field (always curl=0, by construction -- the "conservative field" fact)
fields = {
    "Source: F=(x,y)\ndiv=2, curl=0": lambda x, y: (x, y),
    "Rotation: F=(-y,x)\ndiv=0, curl=2": lambda x, y: (-y, x),
    "Gradient of x^2+y^2: F=(2x,2y)\ndiv=4, curl=0 (irrotational)": lambda x, y: (2*x, 2*y),
}

x, y = np.meshgrid(np.linspace(-2, 2, 12), np.linspace(-2, 2, 12))
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (title, field) in zip(axes, fields.items()):
    U, V = field(x, y)
    ax.quiver(x, y, U, V, color="#4C72B0")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("x"); ax.set_ylabel("y")
    ax.set_aspect('equal')
plt.tight_layout()
plt.show()

print("Source field: arrows point outward everywhere -- positive divergence, no net rotation.")
print("Rotation field: arrows circle counterclockwise -- zero divergence, positive curl.")
print("Gradient field: curl is ALWAYS zero for a gradient field -- this is why conservative")
print("fields (those expressible as grad(f)) are exactly the irrotational ones.")

## 🐍 Implementation from Scratch

We compute divergence, curl, and the Laplacian numerically from their definitions on sampled
grids, and confirm that a gradient field always has zero curl.

In [ ]:
def divergence_2d(P_grid, Q_grid, x_1d, y_1d):
    """div F = dP/dx + dQ/dy, computed on a sampled grid. np.gradient(arr, y_1d, x_1d) on a
    meshgrid('xy')-style array returns [d/dy, d/dx] since axis 0 varies with y."""
    _, dP_dx = np.gradient(P_grid, y_1d, x_1d)
    dQ_dy, _ = np.gradient(Q_grid, y_1d, x_1d)
    return dP_dx + dQ_dy

def curl_2d(P_grid, Q_grid, x_1d, y_1d):
    """(scalar) curl F = dQ/dx - dP/dy."""
    dP_dy, _ = np.gradient(P_grid, y_1d, x_1d)
    _, dQ_dx = np.gradient(Q_grid, y_1d, x_1d)
    return dQ_dx - dP_dy

def laplacian_2d(f_grid, x_1d, y_1d):
    """laplacian f = d^2f/dx^2 + d^2f/dy^2 -- divergence of the gradient."""
    df_dy, df_dx = np.gradient(f_grid, y_1d, x_1d)
    _, d2f_dx2 = np.gradient(df_dx, y_1d, x_1d)
    d2f_dy2, _ = np.gradient(df_dy, y_1d, x_1d)
    return d2f_dx2 + d2f_dy2

x_1d = np.linspace(-2, 2, 60)
y_1d = np.linspace(-2, 2, 60)
Xg, Yg = np.meshgrid(x_1d, y_1d)

# Source field F=(x,y): analytic divergence=2, curl=0 everywhere
div_source = divergence_2d(Xg, Yg, x_1d, y_1d)
curl_source = curl_2d(Xg, Yg, x_1d, y_1d)
print(f"Source field F=(x,y):     mean div = {div_source.mean():.4f} (expect 2), mean curl = {curl_source.mean():.6f} (expect 0)")

# Rotation field F=(-y,x): analytic divergence=0, curl=2 everywhere
div_rot = divergence_2d(-Yg, Xg, x_1d, y_1d)
curl_rot = curl_2d(-Yg, Xg, x_1d, y_1d)
print(f"Rotation field F=(-y,x):  mean div = {div_rot.mean():.4f} (expect 0), mean curl = {curl_rot.mean():.4f} (expect 2)")

# Gradient field F = grad(x^2 + y^2) = (2x, 2y): curl should be ~0 EVERYWHERE (conservative field fact)
curl_grad_field = curl_2d(2 * Xg, 2 * Yg, x_1d, y_1d)
print(f"Gradient field grad(x^2+y^2): max |curl| = {np.max(np.abs(curl_grad_field)):.2e} (expect ~0, up to numerical noise)")

# Laplacian of f(x,y) = x^2 + y^2: analytic answer is 2+2=4 everywhere
lap = laplacian_2d(Xg**2 + Yg**2, x_1d, y_1d)
print(f"\nLaplacian of x^2+y^2 (interior points): {lap[10:-10, 10:-10].mean():.4f}  (expect 4)")

# --- Curl-free does NOT imply conservative in general -- the converse needs a
# simply-connected domain. Counterexample: the vortex field F=(-y,x)/(x^2+y^2), whose domain
# (the plane minus the origin) has a hole a loop can catch on.
def vortex_field(x, y):
    r2 = x ** 2 + y ** 2
    return -y / r2, x / r2

h = 1e-6
def P_v(x, y): return -y / (x ** 2 + y ** 2)
def Q_v(x, y): return x / (x ** 2 + y ** 2)

sample_pts = [(1.0, 0.0), (0.0, 1.5), (-1.2, 0.8), (2.0, -1.0)]
curls = [((Q_v(x + h, y) - Q_v(x - h, y)) / (2 * h) - (P_v(x, y + h) - P_v(x, y - h)) / (2 * h))
         for x, y in sample_pts]
print("\nCurl-free is NOT sufficient for conservative in general -- vortex field F=(-y,x)/(x^2+y^2):")
for (x, y), c in zip(sample_pts, curls):
    print(f"  curl at ({x},{y}): {c:.2e}  (curl-free, same as a true gradient field would look)")

# Yet its circulation around a loop ENCLOSING the hole at the origin is nonzero -- impossible
# for any conservative field, whose circulation around EVERY closed loop must be exactly 0.
n_pts = 200_000
t = np.linspace(0, 2 * np.pi, n_pts, endpoint=False)
dt = 2 * np.pi / n_pts
x_loop, y_loop = np.cos(t), np.sin(t)
dx_loop, dy_loop = -np.sin(t) * dt, np.cos(t) * dt
P_loop, Q_loop = vortex_field(x_loop, y_loop)
circulation_enclosing = np.sum(P_loop * dx_loop + Q_loop * dy_loop)

x_loop2, y_loop2 = 5 + 0.5 * np.cos(t), 5 + 0.5 * np.sin(t)
dx_loop2, dy_loop2 = -0.5 * np.sin(t) * dt, 0.5 * np.cos(t) * dt
P_loop2, Q_loop2 = vortex_field(x_loop2, y_loop2)
circulation_not_enclosing = np.sum(P_loop2 * dx_loop2 + Q_loop2 * dy_loop2)

print(f"Circulation around a loop ENCLOSING the hole at the origin: {circulation_enclosing:.5f}  (2*pi = {2*np.pi:.5f})")
print(f"Circulation around a loop NOT enclosing the hole:           {circulation_not_enclosing:.5f}  (~0)")
print("Nonzero circulation on a loop that can't be shrunk past the hole proves this curl-free")
print("field is NOT conservative -- the missing ingredient was the simply-connected domain.")

## 🤖 Why This Matters for AI

Diffusion models (fully derived in `09.06`) learn a **score function**
$\mathbf{s}(\mathbf{x}) = \nabla_{\mathbf{x}} \log p(\mathbf{x})$ — a vector field pointing
toward regions of higher data density at every point in space, exactly the kind of object this
notebook studies. Since a score function is a *gradient* field by construction, it's always
irrotational (curl-free), a fact this notebook lets you verify directly rather than take on
faith. **Langevin dynamics** generates new samples by literally simulating a particle flowing
through this vector field, nudged by the score plus a little noise:
$\mathbf{x}_{t+1} = \mathbf{x}_t + \eta\,\mathbf{s}(\mathbf{x}_t) + \sqrt{2\eta}\,\boldsymbol{\epsilon}_t$.
Below, we compute the score field of a simple 2D Gaussian mixture, confirm it's curl-free, and
run Langevin dynamics to sample from it — watching particles flow "uphill" along the score
field toward the two modes.

In [ ]:
# Score function of a 2-component Gaussian mixture, verified curl-free, then used to
# drive Langevin dynamics sampling -- the mechanism behind score-based diffusion models.
means = [np.array([-2.0, 0.0]), np.array([2.0, 1.0])]
sigma = 0.8

def score(x):
    # grad_x log p(x) for an equal-weight Gaussian mixture: a responsibility-weighted
    # average of each component's own (simple, Gaussian) score. The responsibilities
    # resp_i = exp(log_unnorm_i) / sum_j exp(log_unnorm_j) are EXACTLY a softmax over each
    # component's unnormalized log-density -- so this needs the same max-subtraction
    # stability guard used for every other softmax in this repo (recall 00.01, 01.10, 06.02).
    # Without it: if x lands far enough from BOTH means (e.g. a particle that wanders off
    # during Langevin dynamics -- see Exercise 3, which asks you to increase eta 5x), every
    # raw exp(-huge_number) underflows to exactly 0.0, total becomes 0.0, and resp becomes
    # [0/0, 0/0] = [nan, nan] -- silently poisoning that particle's position forever after.
    log_unnorm = [-np.sum((x - m) ** 2) / (2 * sigma**2) for m in means]
    m_max = max(log_unnorm)                                  # subtract the max before exponentiating
    comps = [np.exp(lu - m_max) for lu in log_unnorm]         # every argument is now <= 0: never overflows
    total = sum(comps)                                       # >= 1 always (the max term contributes exactly 1)
    resp = [c / total for c in comps]                         # total is never 0 -- no 0/0 possible
    return sum(r * (-(x - m) / sigma**2) for r, m in zip(resp, means))

# Confirm the score field is curl-free (it's a gradient field, so it MUST be, by this
# notebook's own theorem) -- reuse curl_2d on a grid of score vectors
x_1d_s = np.linspace(-5, 5, 60)
y_1d_s = np.linspace(-3, 4, 60)
Xs, Ys = np.meshgrid(x_1d_s, y_1d_s)
Ps = np.zeros_like(Xs); Qs = np.zeros_like(Ys)
for i in range(Xs.shape[0]):
    for j in range(Xs.shape[1]):
        g = score(np.array([Xs[i, j], Ys[i, j]]))
        Ps[i, j], Qs[i, j] = g[0], g[1]
curl_score = curl_2d(Ps, Qs, x_1d_s, y_1d_s)
score_scale = max(np.abs(Ps).max(), np.abs(Qs).max())
print(f"Score field max |curl| (interior): {np.max(np.abs(curl_score[5:-5, 5:-5])):.4f}")
print(f"...compared to the score field's own scale (~{score_scale:.2f}): small, and it shrinks")
print(f"further on a finer grid -- the residual is finite-difference discretization error,")
print(f"not real curl (a true gradient field's curl is EXACTLY zero analytically).")

# --- Regression test: a point FAR from both mixture modes must not produce nan. Both raw
# exponentials would underflow to exactly 0.0 at this distance (>60 sigma from either mean)
# under the unguarded (no max-subtraction) version of this function. ---
far_point = np.array([500.0, 500.0])
score_far = score(far_point)
print(f"\nRegression test: score({far_point.tolist()}) = {score_far}")
print(f"Finite (not nan), as it must be for any x, however far from the mixture's modes: "
      f"{np.all(np.isfinite(score_far))}")
assert np.all(np.isfinite(score_far)), "score() must stay finite even far from every mixture mode"

# Langevin dynamics: let particles flow uphill along the score field, plus noise, and
# watch them settle near the two true mixture modes without ever being TOLD where the modes are
rng = np.random.default_rng(0)
n_particles, eta, n_steps = 150, 0.05, 300
particles = rng.normal(scale=3, size=(n_particles, 2))
trajectory = [particles.copy()]
for _ in range(n_steps):
    grads = np.array([score(p) for p in particles])
    noise = rng.normal(size=particles.shape)
    particles = particles + eta * grads + np.sqrt(2 * eta) * noise
    trajectory.append(particles.copy())

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].streamplot(Xs, Ys, Ps, Qs, color="#4C72B0", density=1.2)
for m in means:
    axes[0].plot(*m, 'r*', markersize=15)
axes[0].set_title("Score field (a gradient field -> curl-free)\nstars = true mixture modes")

axes[1].scatter(trajectory[0][:, 0], trajectory[0][:, 1], alpha=0.4, color="#DD8452", label="start")
axes[1].scatter(particles[:, 0], particles[:, 1], alpha=0.6, color="#55A868", label="after Langevin dynamics")
for m in means:
    axes[1].plot(*m, 'r*', markersize=15)
axes[1].set_title("Langevin dynamics: particles flow along\nthe score field toward the true modes")
axes[1].legend()
plt.tight_layout()
plt.show()

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For $\mathbf{F}(x,y) = (3x^2, -2y)$, compute $\text{div}\,\mathbf{F}$ and
   $\text{curl}\,\mathbf{F}$ by hand. Verify with `divergence_2d`/`curl_2d`. As a sanity check,
   $\text{div}\,\mathbf{F}=6x-2$ is *not* constant here — confirm numerically that it's positive
   for $x>\frac13$, negative for $x<\frac13$, and find the $x$ where the numeric divergence
   crosses zero.

<details>
<summary>💡 Solution</summary>

$P=3x^2$, $Q=-2y$, so $\text{div}\,\mathbf{F}=\partial P/\partial x+\partial Q/\partial y =
6x-2$, and $\text{curl}\,\mathbf{F}=\partial Q/\partial x-\partial P/\partial y = 0-0=0$
everywhere. `divergence_2d`/`curl_2d` reproduce both: at $x\approx1.5$, numeric divergence is
$\approx7.15$ (analytic $6(1.5)-2=7$, matching to within grid resolution), and curl is exactly
$0$ (down to floating-point noise, $\sim10^{-14}$) at every point tested — this field is a net
source ($\text{div}>0$) for $x>\frac13$ and a net sink ($\text{div}<0$) for $x<\frac13$, with no
rotation anywhere. Scanning the numeric divergence along a horizontal slice, its sign flips
right around $x\approx0.31$–$0.33$ (grid-resolution dependent), matching the exact root
$x=\frac13$ where $6x-2=0$.

</details>

### 💭 UNDERSTAND
2. For $f(x,y) = \sin(x)\cos(y)$, the Theory section's Laplacian formula gives
   $\nabla^2 f = -2f$ — the Laplacian is literally a negative scalar multiple of $f$ itself, so
   $\nabla^2 f=0$ exactly where $f=0$. A learner conjectures this is a general pattern: "the
   Laplacian is always zero exactly where the function itself is zero." Test this on
   $g(x,y)=x^2\sin(y)$: find where $g=0$ but $\nabla^2 g\ne0$, and explain why
   $\sin(x)\cos(y)$'s case was special rather than typical.

<details>
<summary>💡 Solution</summary>

$g(x,y)=x^2\sin(y)$ is zero along the entire line $x=0$ (for any $y$), since the $x^2$ factor
vanishes there regardless of $\sin(y)$. But $\nabla^2 g = (2-x^2)\sin(y)$, which *along that
same line* $x=0$ equals $2\sin(y)$ — nonzero for generic $y$ (e.g. at $x=0,y=\pi/2$: $g=0$ but
$\nabla^2g=2$). So the conjecture is false in general: `laplacian_2d` applied to a grid confirms
this numerically (at $x\approx0$, $g\approx0$ everywhere along that column, but the numeric
Laplacian tracks $2\sin(y)$ closely, matching the closed form to within $10^{-3}$). What made
$\sin(x)\cos(y)$ special is that its second $x$- and $y$-derivatives *both* reproduce $-f$
itself with no extra $x$- or $y$-dependence mixed in (a genuine eigenfunction-style relationship,
$\nabla^2 f=\lambda f$ for constant $\lambda=-2$) — that's a property of *this specific*
function, not a general fact about the Laplacian, and $x^2\sin(y)$ (where the second
$x$-derivative introduces a *different* $x$-dependence, $-x^2$ vs. the original's implicit
factor) is a direct counterexample.

</details>

### ✏️ DERIVE
3. Derive the product rule for divergence: for a scalar field $f$ and vector field
   $\mathbf{F}=(P,Q)$, show $\text{div}(f\mathbf{F}) = f\,\text{div}\,\mathbf{F} +
   \nabla f\cdot\mathbf{F}$, starting from the definition $\text{div}(f\mathbf{F}) =
   \partial(fP)/\partial x + \partial(fQ)/\partial y$ and applying the ordinary product rule
   (`02.02`) to each term. Verify your identity numerically on a pair $f,\mathbf{F}$ different
   from any example already in this notebook.

<details>
<summary>💡 Solution outline</summary>

By the product rule applied to each partial derivative separately:
$$\frac{\partial(fP)}{\partial x} = f\frac{\partial P}{\partial x} + P\frac{\partial f}{\partial x},
\qquad \frac{\partial(fQ)}{\partial y} = f\frac{\partial Q}{\partial y} + Q\frac{\partial f}{\partial y}.$$
Adding these two (the definition of $\text{div}(f\mathbf{F})$):
$$\text{div}(f\mathbf{F}) = f\left(\frac{\partial P}{\partial x}+\frac{\partial Q}{\partial y}\right)
+ \left(P\frac{\partial f}{\partial x}+Q\frac{\partial f}{\partial y}\right)
= f\,\text{div}\,\mathbf{F} + \nabla f\cdot\mathbf{F},$$
since the first parenthesized term is exactly $\text{div}\,\mathbf{F}$ and the second is exactly
$\nabla f\cdot\mathbf{F}=(\partial f/\partial x,\partial f/\partial y)\cdot(P,Q)$. Testing on
$f(x,y)=\sin(x)y^2$ and $\mathbf{F}=(x^2y,\ e^xy)$ (over a grid on $[0.5,2.5]^2$): the
directly-computed $\text{div}(f\mathbf{F})$ and the right-hand side
$f\,\text{div}\,\mathbf{F}+\nabla f\cdot\mathbf{F}$, both built from `divergence_2d` and
finite-difference gradients, agree to within about $0.002$ at every interior grid point —
matching well within the finite-difference discretization error, confirming the identity holds
for a genuinely different $(f,\mathbf{F})$ pair, not just the one used to derive it.

</details>

### 🐍 IMPLEMENT
4. Implement a numeric check of **Green's theorem**, $\oint_C\mathbf{F}\cdot d\mathbf{r} =
   \iint_R\text{curl}\,\mathbf{F}\,dA$ (previewed but not verified in the Theory section):
   compute the left side as a line integral around a circle of radius $R$ (sum
   $\mathbf{F}\cdot d\mathbf{r}$ over many points along the circle) for the rotation field
   $\mathbf{F}=(-y,x)$, and the right side as a `curl_2d`-based Riemann sum over the disk the
   circle encloses. State exactly what tolerance the two sides should agree to.

<details>
<summary>✅ How to know you're right</summary>

For $\mathbf{F}=(-y,x)$ (curl $=2$ everywhere, from the Visual Intuition section), both sides
should match the exact value $2\pi R^2$ (twice the disk's area) closely: at $R=1.5$, the
circulation line integral (a fine parametrization, e.g. $200{,}000$ points around the circle)
should land within about $10^{-6}$ of $2\pi(1.5)^2\approx14.137167$, and the `curl_2d`-based
Riemann sum over a masked disk grid (e.g. $800\times800$) should land within about $0.02$ of the
same value — both comfortably inside a $1\%$ tolerance of each other and of the exact answer.
If you additionally try a field with *non-constant* curl (e.g. $\mathbf{F}=(-y^3,x^3)$, whose
curl is $3x^2+3y^2$, not constant), the same two-sided check should still agree to within about
$1\%$ — Green's theorem doesn't require constant curl, so this is a genuine test of the theorem
itself, not an artifact of using a constant-curl field. If your two sides disagree by much more
than that, check the line integral's orientation (Green's theorem assumes $C$ is traversed
counterclockwise) and that `curl_2d`'s grid spacing matches the disk's actual bounds.

</details>

### 🤖 APPLY
5. Real data rarely splits into perfectly equal-weight clusters. Rebuild the AI section's score
   function for an **unequal-weight** 2-component mixture (e.g. weights $0.3$ and $0.7$ for the
   same two means/`sigma`), being careful to fold the log-weights into the responsibility
   computation (`np.log(w) - ...` inside the softmax, not just `-...`). Compare your weighted
   score against the *original* equal-weight `score` function (which implicitly assumes both
   weights are $0.5$) at the exact midpoint between the two means, and explain what the
   equal-weight version gets wrong there.

<details>
<summary>✅ What you should observe</summary>

At the midpoint between the two means $(0, 0.5)$, the correctly-weighted score should come out
clearly nonzero ($\approx(1.25, 0.3125)$ for weights $0.3/0.7$, pulling toward the heavier
$0.7$-weighted mode), with responsibilities $\approx(0.3, 0.7)$ — matching the mixture weights
exactly at this equidistant point, as they should. The original equal-weight `score` function,
applied to the same point, incorrectly reports responsibilities of exactly $(0.5, 0.5)$ and a
score of *exactly* $(0,0)$ — since at the midpoint the two components' distances (and hence
unweighted log-densities) are identical, the equal-weight softmax has no way to break the tie
in the heavier component's favor, and zero score there would incorrectly suggest the midpoint
is some kind of equilibrium point of the *true* (unequal-weight) density, which it is not. Away
from the midpoint the two scores are closer but still numerically different (e.g. at $(-0.5,
0.4)$, differing by $\approx0.29$ in norm) — checking your weighted score against a numeric
gradient of the true log-density (`log_p_weighted`, differentiated with a central difference)
should match to within about $10^{-3}$ at every point tried, confirming the log-weight term is
the only fix needed.

</details>

### 🚀 CHALLENGE
6. A function $f(x,y)$ is **harmonic** if $\nabla^2f=0$ everywhere (Theory section). Harmonic
   functions have a striking property called the **mean value property**: $f$ at the center of
   any circle equals the *average* of $f$ over that circle's boundary, for every radius.
   Verify this numerically for the harmonic function $f(x,y)=x^2-y^2$ (confirm
   $\nabla^2f=0$ first) at an off-center point, using several different circle radii. Then test
   the *non-harmonic* function $g(x,y)=x^2+y^2$ ($\nabla^2g=4\ne0$) the same way, and
   characterize exactly how the boundary average deviates from the center value as a function
   of the radius.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer first confirms $\nabla^2(x^2-y^2)=2-2=0$ (harmonic) and $\nabla^2(x^2+y^2)=2+2=4$
(not harmonic). For the harmonic case, averaging $f(x,y)=x^2-y^2$ over circles of several radii
(e.g. $R=0.5,1,2,3.5$) centered at an arbitrary point like $(1.5,-0.7)$ should reproduce
$f(1.5,-0.7)=1.76$ exactly at *every* radius tested (to floating-point precision, since the
averaging here is over exact trigonometric sample points) — a strong answer explicitly checks
several different radii to confirm the property holds independent of $R$, not just for one lucky
choice. For the non-harmonic case, the boundary average of $g(x,y)=x^2+y^2$ deviates from the
center value by *exactly* $R^2$ at every radius tested (e.g. center value $2.74$, boundary
averages $2.99, 3.74, 6.74, 14.99$ for $R=0.5,1,2,3.5$ — each exactly $R^2$ above center) — a
strong answer identifies this exact quadratic-in-$R$ growth (not just "it's higher than center")
and connects it to the *size* of the nonzero Laplacian: informally, the mean value property's
failure is driven by how much local "bumpiness" ($\nabla^2f\ne0$) exists inside the circle, with
$g$'s constant positive curvature in every direction accumulating a deviation that grows with
the enclosed area ($\propto R^2$) rather than staying flat. A strong answer also connects this
back to the diffusion/score-matching context: since the mean value property is a strong
structural fingerprint of $\nabla^2f=0$, a numerical check like this one could in principle be
used to verify that a learned potential is (or isn't) harmonic somewhere, analogous to this
notebook's own curl-based check for whether a learned vector field is truly a gradient field.

</details>

---

## 📚 Further Reading
- Song & Ermon, ["Generative Modeling by Estimating Gradients of the Data Distribution"](https://arxiv.org/abs/1907.05600) (score-based generative models)
- Strang, *Calculus*, Ch. 15 (Vector Calculus)

---

*Next notebook: [The Calculus of Backpropagation](10_calculus_of_backpropagation.ipynb)*